# Walking a Trace: One Question, End to End (Then Breaking It)

This notebook replays Week 1 the way a debugger would: one full question, turn by turn, watching the prompt grow. Then we attack the loop three ways and watch it recover (or stop). Everything runs on the mock model, so every breakage is reproducible.

Run the setup cell once, then walk the sections in order.

In [ ]:
import re        # pattern-matching: how we find the "Action:" line in model text
import ast       # safe math: lets us read arithmetic as a tree structure without ever running eval()
import operator  # the actual +, -, *, / functions the calculator is allowed to use

# parse_action: the parser half of the contract.
# The prompt tells the model "reply in this exact format"; this function checks
# whether it actually did. Two small regexes: one finds "Action: <name>", the other
# finds "Action Input: <text>". No Action line -> returns None, which the loop
# treats as "the model broke format, send it a correction" (not a crash).
def parse_action(reply):
    """Pull (action_name, action_input) from a model reply.
    Returns None when no Action: line is found -- that is a signal, not a crash."""
    text = reply.strip()
    text = re.sub(r'^```[a-z]*\n?', '', text)   # strip opening fence
    text = re.sub(r'\n?```$', '', text)          # strip closing fence
    m_action = re.search(r'Action:\s*(\w+)', text)
    if not m_action:
        return None
    m_input = re.search(r'Action Input:\s*(.+)', text)
    return m_action.group(1), (m_input.group(1).strip() if m_input else '')

# calculator: a safe arithmetic tool.
# ast.parse turns "12*34" into a tree we can inspect *without running it*.
# We walk the tree and only allow numbers and basic math operators.
# Anything else (try "__import__('os')" below) -> an error *string*, never a crash.
# The model only ever sees the returned string, pasted into an Observation: line.
OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
       ast.Div: operator.truediv, ast.FloorDiv: operator.floordiv,
       ast.Mod: operator.mod, ast.Pow: operator.pow, ast.USub: operator.neg}

def calculator(expr):
    """Arithmetic via ast -- never eval. Anything outside the whitelist
    becomes an error *string*, not an exception."""
    try:
        tree = ast.parse(expr, mode='eval')
    except SyntaxError:
        return "Error: could not parse expression"
    def _eval(node):
        if isinstance(node, ast.Expression):
            return _eval(node.body)
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in OPS:
            return OPS[type(node.op)](_eval(node.left), _eval(node.right))
        if isinstance(node, ast.UnaryOp) and type(node.op) in OPS:
            return OPS[type(node.op)](_eval(node.operand))
        raise ValueError("unsupported expression")
    try:
        return str(_eval(tree))
    except ZeroDivisionError:
        return "Error: division by zero"
    except (ValueError, TypeError, OverflowError):
        return "Error: unsupported expression"

# search + dispatch: tools are just functions with names.
# SEARCH_INDEX stands in for a real search engine: question in, canned answer out.
# TOOLS is the registry -- a plain dict mapping names to functions.
# dispatch() looks up the name the model wrote. Unknown name? That becomes an
# Observation: line telling the model what *is* available, and the loop continues.
# A tool that errors? Also an Observation:. The loop never crashes on a bad action.
SEARCH_INDEX = {
    "boiling point of water in fahrenheit": "The boiling point of water is 212 degrees Fahrenheit.",
    "number of planets": "There are 8 planets in the solar system.",
    "capital of france": "The capital of France is Paris.",
}

def search(query):
    key = query.strip().lower().rstrip('?')
    return SEARCH_INDEX.get(key, f"No results found for '{query}'.")

TOOLS = {"calculator": calculator, "search": search}

def dispatch(action_name, action_input):
    """Route a parsed action to a tool. Misses and errors become
    observations -- the loop never crashes on a bad action."""
    if action_name not in TOOLS:
        return (f"Observation: unknown tool '{action_name}'. "
                f"Available tools: calculator, search.")
    try:
        result = TOOLS[action_name](action_input)
    except Exception as e:
        return f"Observation: tool '{action_name}' raised {type(e).__name__}: {e}."
    return f"Observation: {result}"

# MockModel: a SIMULATED model -- not a real API call.
# It pops pre-written replies from a list, one per turn, like an actor reading
# from a script. That makes every demo in this notebook deterministic, free, and
# runnable anywhere with no API key. The loop cannot tell the difference: it only
# ever sees text, no matter where the text came from.
class MockModel:
    """Deterministic stand-in for ask(): pops canned replies in order.
    The loop cannot tell the difference -- it only ever sees text."""

    def __init__(self, script):
        self.script = list(script)
        self.calls = 0

    def __call__(self, prompt):
        self.calls += 1
        if not self.script:
            return "Thought: I am out of scripted replies.\nFinal Answer: unknown"
        return self.script.pop(0)

# REACT_PROMPT: the contract, written in plain English.
# It tells the model the exact reply format (Thought / Action / Action Input /
# Final Answer) and shows one worked example so the shape is unambiguous.
# build_prompt() slots a new question into the template.
# If the model drifts from the format, fix the prompt first, the parser second.
REACT_PROMPT = """You solve problems by reasoning step by step, then taking actions.
Reply using exactly the format below, nothing else.

Thought: <your reasoning>
Action: <one of: calculator, search>
Action Input: <input for the action>
... (repeat Thought/Action/Action Input as needed)
Thought: <final reasoning>
Final Answer: <the answer>

Example:
Question: What is 12*34 plus 8?
Thought: I need to compute 12*34 first.
Action: calculator
Action Input: 12*34
Observation: 408
Thought: Now add 8.
Action: calculator
Action Input: 408+8
Observation: 416
Thought: I have the answer.
Final Answer: 416

Question: {question}
"""

def build_prompt(question):
    return REACT_PROMPT.format(question=question)

# run(): the whole agent in one function. Five lines of logic wearing a trench coat:
#   1. ask the model (here, the mock) with the current prompt
#   2. if it wrote "Final Answer:", stop and return the answer
#   3. parse the Action: line (None -> append a format-correction observation)
#   4. dispatch the tool, wrap the result as an Observation: line
#   5. append reply + observation to the prompt, repeat (safety cap: 8 turns)
def run(question, model, max_iterations=8, track_prompt_len=False):
    """The whole agent: ask, parse, dispatch, append observation, repeat."""
    prompt = build_prompt(question)
    trace, prompt_lens = [], []
    for _ in range(max_iterations):
        if track_prompt_len:
            prompt_lens.append(len(prompt))
        reply = model(prompt)
        if "Final Answer:" in reply:
            answer = reply.split("Final Answer:")[1].strip()
            trace.append({"answer": answer})
            return {"answer": answer, "trace": trace, "prompt_lens": prompt_lens}
        parsed = parse_action(reply)
        if parsed is None:
            obs = ("Observation: I couldn't parse an action from your last response. "
                   "Reply using exactly the format: Thought: ... Action: ... Action Input: ...")
            trace.append({"raw": reply, "observation": obs})
        else:
            name, action_input = parsed
            obs = dispatch(name, action_input)
            thought = re.search(r'Thought:\s*(.+)', reply)
            trace.append({"thought": thought.group(1).strip() if thought else "",
                          "action": f"{name}({action_input})",
                          "observation": obs})
        prompt += "\n" + reply + "\n" + obs + "\n"
    return {"answer": "max iterations reached", "trace": trace, "prompt_lens": prompt_lens}


def print_trace(result):
    for i, t in enumerate(result["trace"], 1):
        if "answer" in t:
            print(f"--- Turn {i}: FINAL ---\nFinal Answer: {t['answer']}\n")
        elif "raw" in t:
            print(f"--- Turn {i}: MALFORMED ---\nRaw reply: {t['raw']}\n{t['observation']}\n")
        else:
            print(f"--- Turn {i} ---\nThought: {t['thought']}\n"
                  f"Action: {t['action']}\n{t['observation']}\n")
    print("Answer:", result["answer"])

> **Simulated model ahead.** The next cells use `MockModel`, a stand-in that replays pre-written replies instead of calling a real API. Everything the loop does with those replies (parse, dispatch, recover from garbage) is identical to production. Only the *source* of the text is fake, which is what makes every demo reproducible.

## The Question

"What is the boiling point of water in Fahrenheit, multiplied by the number of planets?" It needs both tools: search for the facts, calculator for the math.

Watch two things as the trace prints: the prompt length after every turn, and how ordinary each step looks. There is no magic step. There never was.

In [ ]:
script = [
    "Thought: I need the boiling point of water in Fahrenheit.\nAction: search\nAction Input: boiling point of water in Fahrenheit",
    "Thought: Now I need the number of planets.\nAction: search\nAction Input: number of planets",
    "Thought: Now multiply 212 by 8.\nAction: calculator\nAction Input: 212*8",
    "Thought: I have everything I need.\nFinal Answer: 1696",
]

result = run(
    "What is the boiling point of water in Fahrenheit, multiplied by the number of planets?",
    MockModel(script),
    track_prompt_len=True,
)
print_trace(result)
print("Prompt length per iteration:", result["prompt_lens"])
print("\nThe prompt grew by ~%d characters per turn. That growth never stops. (Week 3.)"
      % ((result["prompt_lens"][-1] - result["prompt_lens"][0]) // max(len(result["prompt_lens"]) - 1, 1)))

## Breakage 1: Garbage In, Correction Out

The mock emits nonsense on its first turn ("Hmm, let me think... 42!"). The parser returns `None`. The loop does not crash. It appends a correction observation and hands the turn back.

Watch the trace: the recovery is visible as an observation, not an exception. This is the path the explain test makes you narrate.

In [ ]:
script_b1 = [
    "Hmm, let me think... 42!",
    "Thought: That was not the format. I need the boiling point of water in Fahrenheit.\nAction: search\nAction Input: boiling point of water in Fahrenheit",
    "Thought: Now multiply by the number of planets, which is 8.\nAction: calculator\nAction Input: 212*8",
    "Thought: Done.\nFinal Answer: 1696",
]

print_trace(run(
    "What is the boiling point of water in Fahrenheit, multiplied by the number of planets?",
    MockModel(script_b1)))

## Breakage 2: The Tool That Does Not Exist

The mock asks for `web_search`. There is no `web_search` in the registry, and that is fine: the miss becomes an observation naming the tools that *do* exist. The model reads it and retries with `search`.

A malformed reply and an unknown tool take the same road home: converted to text, fed back, loop continues.

In [ ]:
script_b2 = [
    "Thought: I'll search the web for this.\nAction: web_search\nAction Input: boiling point of water in Fahrenheit",
    "Thought: web_search is not available. I will use search instead.\nAction: search\nAction Input: boiling point of water in Fahrenheit",
    "Thought: Now multiply by 8.\nAction: calculator\nAction Input: 212*8",
    "Thought: Done.\nFinal Answer: 1696",
]

print_trace(run(
    "What is the boiling point of water in Fahrenheit, multiplied by the number of planets?",
    MockModel(script_b2)))

## Breakage 3: The Iteration Cap

An unanswerable question ("What is the airspeed velocity of an unladen swallow?") with a stub search that has no entry. The loop tries, gets "no results" observations, and at 8 iterations it stops cleanly with "max iterations reached".

The cap is the safety net. You verified it fires; you did not trust it.

In [ ]:
script_b3 = [
    "Thought: I need the airspeed velocity of an unladen swallow.\nAction: search\nAction Input: airspeed velocity of an unladen swallow",
] * 8

print_trace(run(
    "What is the airspeed velocity of an unladen swallow?",
    MockModel(script_b3)))

## Compare With Your Real Log

These three breakages are the reproducible, mock-driven version of your `day-05-breakage-log.md`. Pull up the real log and compare: what did the live run catch that the mock cannot? (Real latency, real model stubbornness, real API errors.) The mock proves the loop's logic; the log proves it survived contact with reality. You need both.

## The Explain Test

Out loud, no notes, narrate all four:

1. **A normal two-step turn:** prompt -> model text -> parse -> dispatch -> observation appended -> repeat -> Final Answer.
2. **A malformed tool call:** garbage in -> parser returns None -> correction observation -> model retries.
3. **An unknown tool:** dispatch miss -> observation naming real tools -> model retries.
4. **The iteration cap:** unanswerable question -> 8 turns -> clean stop, no crash.

If all four are crisp, Week 1 is done. And now you have the template: every sprint week gets one of these.